<a href="https://colab.research.google.com/github/Sathvikabanavath/GenAI-Tasks/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q faiss-cpu sentence-transformers transformers accelerate

In [ ]:
documents = [
    "The capital of France is Paris. It is known for the Eiffel Tower.",
    "Python is a high-level programming language created by Guido van Rossum.",
    "The Great Wall of China is over 13,000 miles long.",
    "The Earth revolves around the Sun and takes about 365 days to complete one orbit.",
    "Albert Einstein developed the theory of relativity.",
    "Water boils at 100 degrees Celsius at sea level.",
    "The human heart has four chambers.",
    "Mount Everest is the highest mountain in the world.",
]

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load embedding model
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Create embeddings
embeddings = embedder.encode(documents, convert_to_numpy=True)

# Build FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

In [ ]:
def retrieve_chunks(query, k=3):
    query_embedding = embedder.encode([query])
    distances, indices = index.search(query_embedding, k)
    return [documents[i] for i in indices[0]]

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
def rag_answer(question, k=3):
    # Retrieve context
    contexts = retrieve_chunks(question, k)
    context_text = " ".join(contexts)

    # Prompt
    prompt = f"""
    Answer the question using only the context below.

    Context:
    {context_text}

    Question:
    {question}

    Answer:
    """

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
questions = [
    "What is the capital of France?",
    "Who created Python?",
    "How long does the Earth take to orbit the Sun?",
    "Who developed the theory of relativity?",
    "What is the highest mountain in the world?"
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {rag_answer(q)}")
    print("-" * 60)